# 02 · Run attribution + steering
Loads Qwen2.5-Omni-7B Thinker (bf16, ~16 GB of weights; experiments were run on an NVIDIA A100 80GB) and, for every item,
computes the contrastive attribution and generates answers for every (mode, K, alpha).
Runs are resumable per (dataset, select score, mode, K, alpha, item). On CUDA OOM set `config.FREE_CACHE = True` and `config.BATCH_ALPHAS = False`.

In [ ]:
# Storage locations: set them BEFORE importing caga (see caga/paths.py).
# CAGA_BASE defaults to ~/caga; point it at a disk with ~60 GB free.
import os, sys
# os.environ["CAGA_BASE"] = "/path/to/large/disk/caga"

# Make the repository importable whether the notebook runs from the repo root or from notebooks/.
for cand in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(cand, "caga")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break

### Configuration
All settings live in `caga/config.py`; override them here. See `REPORT.md` for what each option means.

In [ ]:
from caga import config

DATASETS = ["mmar"]
SELECT_SCORES = ["contrast"]            # any of config.SELECT_SCORES; run one after another
config.MODES = ["final", "audio", "text", "audio_text", "per_token"]
config.K_LIST = [10000]
config.ALPHAS = [-1.0, -0.3, 0.0, 0.3, 1.0]
config.PERTOKEN_TOPK = False

### Optional: self-tests (once per environment)

In [ ]:
from caga import selftest
# selftest.selftest_alpha_batch("mmau_mini")      # batched alphas must MATCH sequential generation
# selftest.selftest_fast_attr("mmau_mini")        # only if you want to try config.FAST_ATTR = True
# selftest.time_one_item("mmar")

### Run

In [ ]:
from caga.experiment import run_steering, evaluate

for sel in SELECT_SCORES:
    config.SELECT_SCORE = sel
    print(f"\n########## SELECT_SCORE = {sel} ##########")
    run_steering(DATASETS)
    evaluate(DATASETS)

### Save tables (one Excel file per dataset in `RUNS_BASE`)

In [ ]:
from caga.results import save_excel
for ds in DATASETS:
    df = save_excel(ds)
df